AF 8-2025

# Final 

This time we want to confirm that: 

1) video games are the "biggest" increase for 15-25 and for 15-30
2) run the charts for 
    Tv with others
    Tv alone
    Gaming alone
    Gaming with others
    Socializing
    Exercise
    Sleep (separate)


In [1]:
import pandas as pd
import numpy as np

from atus_utils import PROCESSED, load_table, add_weight, code_social_activities, weighted_average_pivot

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

MAX_AGES = [25, 30]

In [2]:
# READ IN THE FILES -- only the tables and columns we use

sum_variables = ["TUCASEID", # ID
                "TUYEAR", # Year
                "TESCHENR", # School enrollment
                "TRSPPRES", # Presence of spouse
                "TUDIARYDAY", # day of week
                "TESEX", # Sex
                "TRERNWA", # Weekly earnings 
                "TEHRUSLT", # Usual weekly hours of work 
                "TESCHLVL", # school level
                "weight", # Weight
                "TEAGE", # Age 
                "PTDTRACE", # Race
                "PEHSPNON", # Hispanic Origin
                "PEEDUCA", # Educational attainment
                "GTMETSTA", # new metro status
                "GEMETSTA", # old metro status
                "TELFS" # employment status
]

a = {}

a['sum'] = load_table('sum', [v for v in sum_variables if v != 'weight'] + ['TUFNWGTP', 'TU20FWGT'])
a['act'] = load_table('act', ['TUCASEID', 'TUACTIVITY_N', 'TUACTDUR24', 'TRCODEP', 'TRTIER1P', 'TRTIER2P'])

In [3]:
# TUFNWGTP is -1 for all of 2020, so 2020 uses TU20FWGT
a['sum'] = add_weight(a['sum'])

In [4]:
a['sum'].head()

,TUCASEID,TUYEAR,TESCHENR,TRSPPRES,TUDIARYDAY,TESEX,TRERNWA,TEHRUSLT,TESCHLVL,TEAGE,PTDTRACE,PEHSPNON,PEEDUCA,GTMETSTA,GEMETSTA,TELFS,TUFNWGTP,TU20FWGT,weight
0,20030100013280,2003,-1,1,6,1,66000,30,-1,60,2,2,44,-1,1,2,8.155463e+06,-1.0,8.155463e+06
1,20030100013344,2003,2,1,7,2,20000,30,-1,41,1,2,40,-1,2,1,1.735323e+06,-1.0,1.735323e+06
2,20030100013352,2003,2,1,7,2,20000,12,-1,26,1,2,41,-1,1,2,3.830527e+06,-1.0,3.830527e+06
3,20030100013848,2003,2,1,5,2,-1,-1,-1,36,2,2,39,-1,2,4,6.622023e+06,-1.0,6.622023e+06
4,20030100014165,2003,-1,1,5,1,-1,80,-1,51,1,2,45,-1,2,1,3.068387e+06,-1.0,3.068387e+06


In [ ]:
with pd.ExcelWriter(PROCESSED / 'nb_5_final_F.xlsx') as writer:

    for MAX_AGE in MAX_AGES:

        sample = a['sum'][(a['sum']['TEAGE']<MAX_AGE) & (a['sum']['TUYEAR']>=2004)]
        sample = sample[sum_variables]

        SAMPLE_IDS = sample.TUCASEID

        activities = a['act'][a['act'].TUCASEID.isin(SAMPLE_IDS)].copy()
        activities['description'] = code_social_activities(activities)

        acts = weighted_average_pivot(activities, sample, 'weight', ['TUYEAR'], ['TESEX'], ['description'], 'TUACTDUR24')
        acts_trcodep = weighted_average_pivot(activities, sample, 'weight', ['TUYEAR'], ['TESEX'], ['TRCODEP'], 'TUACTDUR24')

        acts_sig = acts_trcodep[acts_trcodep[2024]>=0.5].sort_values('percent_change_ratio', ascending = False)

        acts.to_excel(writer, sheet_name = f"displacement_15-{MAX_AGE-1}", index = None)
        acts_sig.to_excel(writer, sheet_name = f"above_0.5_hours_per_week_15-{MAX_AGE-1}", index = None)

In [6]:
activities.head()

,TUCASEID,TUACTIVITY_N,TUACTDUR24,TRCODEP,TRTIER1P,TRTIER2P,description
412638,20040101040079,1,240,10101,1,101,Sleeping
412639,20040101040079,2,30,20101,2,201,all other activities
412640,20040101040079,3,5,180701,18,1807,all other activities
412641,20040101040079,4,25,70101,7,701,all other activities
412642,20040101040079,5,5,180701,18,1807,all other activities


In [7]:
a['sum'].head()

,TUCASEID,TUYEAR,TESCHENR,TRSPPRES,TUDIARYDAY,TESEX,TRERNWA,TEHRUSLT,TESCHLVL,TEAGE,PTDTRACE,PEHSPNON,PEEDUCA,GTMETSTA,GEMETSTA,TELFS,TUFNWGTP,TU20FWGT,weight
0,20030100013280,2003,-1,1,6,1,66000,30,-1,60,2,2,44,-1,1,2,8.155463e+06,-1.0,8.155463e+06
1,20030100013344,2003,2,1,7,2,20000,30,-1,41,1,2,40,-1,2,1,1.735323e+06,-1.0,1.735323e+06
2,20030100013352,2003,2,1,7,2,20000,12,-1,26,1,2,41,-1,1,2,3.830527e+06,-1.0,3.830527e+06
3,20030100013848,2003,2,1,5,2,-1,-1,-1,36,2,2,39,-1,2,4,6.622023e+06,-1.0,6.622023e+06
4,20030100014165,2003,-1,1,5,1,-1,80,-1,51,1,2,45,-1,2,1,3.068387e+06,-1.0,3.068387e+06


In [8]:
sample = a['sum'][(a['sum']['TEAGE']<25) & (a['sum']['TUYEAR']>=2004)]


In [9]:
sample.groupby(['TESEX', 'TUYEAR'])['TUCASEID'].count()

TESEX  TUYEAR
1      2004      763
       2005      717
       2006      733
       2007      676
       2008      683
       2009      670
       2010      708
       2011      626
       2012      608
       2013      542
       2014      533
       2015      481
       2016      466
       2017      396
       2018      378
       2019      361
       2020      373
       2021      354
       2022      277
       2023      281
       2024      273
2      2004      832
       2005      812
       2006      860
       2007      755
       2008      714
       2009      702
       2010      724
       2011      694
       2012      591
       2013      557
       2014      542
       2015      493
       2016      473
       2017      435
       2018      379
       2019      358
       2020      324
       2021      351
       2022      266
       2023      299
       2024      259
Name: TUCASEID, dtype: int64

In [10]:
a['sum'].groupby('TUYEAR')['TUCASEID'].count()

TUYEAR
2003    20720
2004    13973
2005    13038
2006    12943
2007    12248
2008    12723
2009    13133
2010    13260
2011    12479
2012    12443
2013    11385
2014    11592
2015    10905
2016    10493
2017    10223
2018     9593
2019     9435
2020     8782
2021     9087
2022     8136
2023     8548
2024     7669
Name: TUCASEID, dtype: int64